## 9.5 文書検索モデルとChatGPTを組み合わせる

### 9.5.1 検索モデルの準備

In [2]:
!pip install 'datasets<4.0.0' openai==0.27 tiktoken 'transformers[ja]<4.41.0'  faiss-cpu

In [3]:
from datasets import load_dataset
from transformers import pipeline

dataset_name = "llm-book/aio-passages-bpr-bert-base-japanese-v3"
passage_dataset = load_dataset(dataset_name, split="train")

encoder_model_name = "llm-book/bert-base-japanese-v3-bpr-question-aio"
encoder_pipeline = pipeline(
    "feature-extraction", model=encoder_model_name, device="cuda:0"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
print(passage_dataset)

Dataset({
    features: ['id', 'pageid', 'revid', 'text', 'section', 'title', 'embeddings'],
    num_rows: 4288198
})


In [5]:
print(passage_dataset[0]["embeddings"])

[133, 162, 145, 21, 151, 215, 254, 119, 214, 80, 4, 189, 177, 53, 100, 115, 68, 177, 70, 103, 41, 90, 127, 227, 113, 27, 71, 148, 92, 162, 176, 133, 105, 99, 16, 16, 52, 70, 132, 46, 2, 32, 211, 149, 29, 103, 53, 233, 29, 199, 124, 65, 178, 90, 60, 32, 201, 114, 214, 132, 60, 254, 216, 249, 184, 57, 119, 181, 23, 253, 121, 83, 63, 115, 141, 73, 90, 0, 239, 225, 194, 248, 16, 108, 66, 215, 124, 248, 0, 26, 214, 78, 52, 118, 115, 194]


In [6]:
print(dir(passage_dataset))

['_TF_DATASET_REFS', '__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getitems__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_build_local_temp_path', '_check_index_is_initialized', '_data', '_estimate_nbytes', '_fingerprint', '_format_columns', '_format_kwargs', '_format_type', '_generate_tables_from_cache_file', '_generate_tables_from_shards', '_get_cache_file_path', '_get_output_signature', '_getitem', '_indexes', '_indices', '_info', '_map_single', '_new_dataset_with_indices', '_output_all_columns', '_push_parquet_shards_to_hub', '_save_to_disk_single', '_select_contiguous', '_select_with_indices_mapping', '_split', 'add_colum

#### `passage_dataset["embeddings"]`を`Faiss`ライブラリのインデックスに格納する

In [8]:
import faiss
import numpy as np
from tqdm import tqdm

# インデックスの初期化
embed_size = encoder_pipeline.model.config.hidden_size
faiss_index = faiss.IndexBinaryIDMap2(
    faiss.IndexBinaryFlat(embed_size)
)

with tqdm(total=len(passage_dataset)) as pbar:
    i = 0
    for batch in passage_dataset.iter(batch_size=512):
        bs = len(batch["embeddings"])

        # 埋め込みをインデックスに適したdtypeへと変換
        batch_embeddings = np.array(
            batch["embeddings"], dtype=np.uint8
        )
        batch_indices = np.arange(i, i + bs, dtype=np.int64)

        # 埋め込みをインデックスに格納
        faiss_index.add_with_ids(batch_embeddings, batch_indices)

        pbar.update(n=bs)
        i += bs

100%|██████████| 4288198/4288198 [03:51<00:00, 18496.58it/s]


In [9]:
import torch

def embed_questions(questions: list[str]) -> np.ndarray:
    """質問文を実数ベクトルに変換"""
    output_tensors = encoder_pipeline(questions, return_tensors="pt")
    embeddings = np.stack([
        t.squeeze(0)[0].numpy().astype(np.float32)
        for t in output_tensors
    ])
    return embeddings

def binarize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    """実数ベクトルをバイナリベクトルに変換"""
    # 0未満の値を0に、0以上の値を1に変換
    binary_embeddings = np.where(embeddings < 0, 0, 1)
    # バイト単位でまとめてuint8に変換
    packed_binary_embeddings = np.packbits(binary_embeddings, axis=1)
    return packed_binary_embeddings

In [10]:
q_embed = embed_questions(["日本で一番高い山は何？"])
binary_q_embed = binarize_embeddings(q_embed)
print(binary_q_embed)

[[141 210 243   2 153 138 188 209 212 205 166 253 146  26  12 135 215 208
    0 233 223  17  43   5 198 163   1 116 111  63 138 212 106 118 126  84
   31 166  23 194 190 164 201 165 139  83 132  46  86  11  97  24  81 227
   12  46 128 115  17  49  22 224 101  16 252 100 229  33  92  77 216 119
  176  62 252  97  16 110 203  36 195 208  32 170 138 159 255  83  83 122
   64 166 100 246  83  44]]
